In [ ]:
import pandas as pd
import numpy as np 
from os import listdir
from os.path import isfile, join

import math


In [ ]:
df_brute = pd.read_csv("../data/data_cleaned/Pseudonymisation_provisoire_geocoded_dpt_reg.csv",sep=";")
df_revenus = pd.read_csv("../analyse_clinique/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]

In [ ]:
df_revenus[df_revenus["DISP_MED20"]=="ns"] = np.nan
df_revenus[df_revenus["DISP_MED20"]=="nd"] = np.nan

df_revenus["DISP_MED20"] = df_revenus["DISP_MED20"].astype(float)
# df_brute.loc[:,"CODE_IRIS"] = df_brute["CODE_IRIS"].str.strip()
df_revenus = df_revenus.rename({"IRIS":"CODE_IRIS"})
df_brute = df_brute.dropna(subset="CODE_IRIS")
df_brute_rev = df_brute.merge(df_revenus,how="left",left_on="CODE_IRIS",right_on="IRIS")

# Patients

## Biais unique 

In [ ]:
files_biais_unique = [f for f in listdir("data_with_biais/patients_unique_biais")]
dict_df_unique = {}

col_to_drop = [ 'adresse_has_RESIDENCE_init', 'adresse_has_CHEZ_init',
       'adresse_has_BAT_init', 'adresse_has_APPT_init', 'adresse_has_MME_init',
       'adresse_has_MR_init', 'adresse_has_RES_init',
       'adresse_has_HOPITAL_init', 'adresse_has_MAISON_init',
       'adresse_has_RETRAITE_init', 'adresse_has_CENTRE_init',
       'adresse_has_HOTEL_init', 'adresse_has_QUARTIER_init']


# for file in files_biais_unique:
#     nom_df = file[:-4]
#     df = pd.read_csv(join("data_with_biais/patients_unique_biais",file),sep=";",dtype=str)
#     dict_df_unique[nom_df] = df.drop(col_to_drop, axis=1)

df_APPT = pd.read_csv("data_with_biais/patients_unique_biais/df_patients_APPT.csv", sep=";",dtype=str)
df_APPT = df_APPT.drop("adresse_has_APPT_init", axis=1)


### Ajout des tag et jointure avec df revenus 

In [ ]:
# dict_df_unique_tag = {}
# for nom_df,df in dict_df_unique.items():
#     nom_tag = nom_df.split("_")[2]
#     df["adresse"] = df["adresse"].astype(str)
#     df[f"adresse_has_{nom_tag}_init"] = False
#     for i in range(len(df)):
#         if nom_tag in df.at[i,"adresse"].split() :
#             df.at[i,f"adresse_has_{nom_tag}_init"] = True 
#     df = df.dropna(subset="CODE_IRIS")
#     df_w_revenus = df.merge(df_revenus, how="left",left_on="CODE_IRIS",right_on="IRIS")
#     dict_df_unique_tag[nom_df] = df_w_revenus


nom_tag ="APPT"
df_APPT["adresse"] = df_APPT["adresse"].astype(str)
df_APPT[f"adresse_has_{nom_tag}_init"] = False
for i in range(len(df_APPT)):
    if nom_tag in df_APPT.at[i,"adresse"].split() :
        df_APPT.at[i,f"adresse_has_{nom_tag}_init"] = True 
df_APPT = df_APPT.dropna(subset="CODE_IRIS")
df_w_revenus = df_APPT.merge(df_revenus, how="left",left_on="CODE_IRIS",right_on="IRIS")


### Ajout de la distance et différence de revenu 

In [ ]:
def calculer_distance_haversine(lat1, lon1, lat2, lon2):

    try : 
        # Rayon de la Terre en kilomètres
        R = 6371.0

        # Conversion des degrés en radians
        dLat = math.radians(lat2 - lat1)
        dLon = math.radians(lon2 - lon1)
        rLat1 = math.radians(lat1)
        rLat2 = math.radians(lat2)

        # Formule de Haversine
        a = math.sin(dLat / 2)**2 + math.cos(rLat1) * math.cos(rLat2) * math.sin(dLon / 2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        # Distance totale
        distance = R * c
        return distance
    except Exception as e : 
        print(f"Erreur lors du calcul de la distance : {e}")
        return np.nan

In [ ]:
list_id_pat_unique = {}
dict_df_unique_dist = {}
for nom_df,df in dict_df_unique_tag.items():
    nom_tag = nom_df.split("_")[2]
    list_id_tag = []

    # PREP DATA 
    df["DISP_MED20"] = df["DISP_MED20"].astype(float)
    df_brute_rev["DISP_MED20"] = df_brute_rev["DISP_MED20"].astype(float)

    df["y"] = df["y"].astype(float)
    df["x"] = df["x"].astype(float)

    df_brute_rev["x"] = df_brute_rev["x"].astype(float)
    df_brute_rev["y"] = df_brute_rev["y"].astype(float)

    df_cleaned = df.dropna(subset=["x","y","DISP_MED20"])
    df_brute_rev = df_brute_rev.dropna(subset=["x","y","DISP_MED20"])
    df_cleaned.reset_index(drop=True, inplace = True)
    df_brute_rev.reset_index(drop=True,inplace=True)

    indice_communs = set(df_cleaned.index) & set(df_brute_rev.index)

    df_cleaned = df_cleaned[df_cleaned.index.isin(indice_communs)]
    df_brute_rev = df_brute_rev[df_brute_rev.index.isin(indice_communs)]

    for i in df_cleaned.index:
        if df_cleaned.at[i,f"adresse_has_{nom_tag}_init"] == True :
            list_id_tag.append(i)

        lon1 = df_cleaned.at[i,"x"]
        lat1 = df_cleaned.at[i,"y"]
        lon2 = df_brute_rev.at[i,"x"]
        lat2 = df_brute_rev.at[i,"y"]

        if pd.isna(lon1) or pd.isna(lat1) or pd.isna(lon2) or pd.isna(lat2): 
            print(i)

        df_cleaned.loc[i,f"distance_{nom_tag}"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)
        df_cleaned.loc[i,f"diff_revenu_{nom_tag}"] = pd.to_numeric(df_cleaned.at[i,"DISP_MED20"]) - pd.to_numeric(df_brute_rev.at[i,"DISP_MED20"])
        
    dict_df_unique_dist[nom_df]=df_cleaned
    list_id_pat_unique[nom_tag] = list_id_tag 
    print(f"{nom_df} modifié")


In [ ]:
## APPT : 
nom_tag ="APPT"
df = df_w_revenus
list_id_tag = []

# PREP DATA 
df["DISP_MED20"] = df["DISP_MED20"].astype(float)
df_brute_rev["DISP_MED20"] = df_brute_rev["DISP_MED20"].astype(float)

df["y"] = df["y"].astype(float)
df["x"] = df["x"].astype(float)

df_brute_rev["x"] = df_brute_rev["x"].astype(float)
df_brute_rev["y"] = df_brute_rev["y"].astype(float)

df_cleaned = df.dropna(subset=["x","y","DISP_MED20"])
df_brute_rev = df_brute_rev.dropna(subset=["x","y","DISP_MED20"])
df_cleaned.reset_index(drop=True, inplace = True)
df_brute_rev.reset_index(drop=True,inplace=True)

indice_communs = set(df_cleaned.index) & set(df_brute_rev.index)

df_cleaned = df_cleaned[df_cleaned.index.isin(indice_communs)]
df_brute_rev = df_brute_rev[df_brute_rev.index.isin(indice_communs)]

for i in df_cleaned.index:
    if df_cleaned.at[i,f"adresse_has_{nom_tag}_init"] == True :
        list_id_tag.append(i)

    lon1 = df_cleaned.at[i,"x"]
    lat1 = df_cleaned.at[i,"y"]
    lon2 = df_brute_rev.at[i,"x"]
    lat2 = df_brute_rev.at[i,"y"]

    if pd.isna(lon1) or pd.isna(lat1) or pd.isna(lon2) or pd.isna(lat2): 
        print(i)

    df_cleaned.loc[i,f"distance_{nom_tag}"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)
    df_cleaned.loc[i,f"diff_revenu_{nom_tag}"] = pd.to_numeric(df_cleaned.at[i,"DISP_MED20"]) - pd.to_numeric(df_brute_rev.at[i,"DISP_MED20"])
    
# dict_df_unique_dist[nom_df]=df_cleaned
# list_id_pat_unique[nom_tag] = list_id_tag 

In [ ]:
df_cleaned.to_csv("data_with_biais/patients_unique_biais/df_patients_APPT_dist.csv")

In [ ]:
for nom_df, df in dict_df_unique_dist.items():

    df.to_csv(f"data_with_biais/patients_unique_biais/{nom_df}_dist.csv",sep=";")

## Biais joint 

In [ ]:
files_biais_join = [f for f in listdir("data_with_biais/patients_join_biais")]
dict_df_join = {}

for file in files_biais_join:
    nom_df = file[:-4]
    df = pd.read_csv(join("data_with_biais/patients_join_biais",file),sep=";",dtype=str)
    dict_df_join[nom_df]= df
    # dict_df_unique[nom_df] = df.drop(col_to_drop, axis=1)


### Ajout des tag et jointure avec revenus

In [ ]:
dict_df_join_tag = {}
for nom_df,df in dict_df_join.items():
    nom_tag = nom_df.split("_")[2]+'_'+ nom_df.split("_")[3]
    tag_1 = nom_df.split("_")[2]
    tag_2 = nom_df.split("_")[3]
    df["adresse"] = df["adresse"].astype(str)
    df[f"adresse_has_{nom_tag}_init"] = False
    for i in range(len(df)):
        if tag_1 in df.at[i,"adresse"].split() or tag_2 in df.at[i,"adresse"].split():
            df.at[i,f"adresse_has_{nom_tag}_init"] = True 
    df = df.dropna(subset="CODE_IRIS")
    df_w_revenus = df.merge(df_revenus, how="left",left_on="CODE_IRIS",right_on="IRIS")
    dict_df_join_tag[nom_df] = df_w_revenus


In [ ]:
list_id_pat_join = {}
dict_df_join_dist = {}
for nom_df,df in dict_df_join_tag.items():
    nom_tag = nom_df.split("_")[2]+'_'+ nom_df.split("_")[3]

    list_id_tag = []

    # PREP DATA 
    df["DISP_MED20"] = df["DISP_MED20"].astype(float)
    df_brute_rev["DISP_MED20"] = df_brute_rev["DISP_MED20"].astype(float)

    df["y"] = df["y"].astype(float)
    df["x"] = df["x"].astype(float)

    df_brute_rev["x"] = df_brute_rev["x"].astype(float)
    df_brute_rev["y"] = df_brute_rev["y"].astype(float)

    df_cleaned = df.dropna(subset=["x","y","DISP_MED20"])
    df_brute_rev = df_brute_rev.dropna(subset=["x","y","DISP_MED20"])

    id_cleaned = df_cleaned.index 
    id_cleaned_rev = df_brute_rev.index 
    df_brute_rev_cleaned = df_brute_rev[(df_brute_rev.index.isin(id_cleaned)) & (df_brute_rev.index.isin(id_cleaned_rev))]
    df_cleaned = df_cleaned[(df_cleaned.index.isin(id_cleaned)) & (df_cleaned.index.isin(id_cleaned_rev))]


    for i in df_cleaned.index:
        if df_cleaned.at[i,f"adresse_has_{nom_tag}_init"] == True :
            list_id_tag.append(i)

        lon1 = df_cleaned.at[i,"x"]
        lat1 = df_cleaned.at[i,"y"]
        lon2 = df_brute_rev_cleaned.at[i,"x"]
        lat2 = df_brute_rev_cleaned.at[i,"y"]

        df_cleaned.loc[i,f"distance_{nom_tag}"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)
        df_cleaned.loc[i,f"diff_revenu_{nom_tag}"] = pd.to_numeric(df_cleaned.at[i,"DISP_MED20"]) - pd.to_numeric(df_brute_rev_cleaned.at[i,"DISP_MED20"])

    dict_df_join_dist[nom_df]=df_cleaned
    list_id_pat_join[nom_tag] = list_id_tag 
    print(f"{nom_df} modifié")


In [ ]:
for nom_df, df in dict_df_join_dist.items():

    df.to_csv(f"data_with_biais/patients_join_biais/{nom_df}_dist.csv",sep=";")

# Références 

In [ ]:
df_ref_brute = pd.read_csv("gendarmerie/gendarmerie_geocoded.csv", sep=";")

## Biais unique 

In [ ]:
files_ref_biais_unique = [f for f in listdir("data_with_biais/ref_unique_biais")]
dict_df_ref_unique = {}

for file in files_ref_biais_unique:
    nom_df = file[:-4]
    df = pd.read_csv(join("data_with_biais/ref_unique_biais",file),sep=";",dtype=str)
    dict_df_ref_unique[nom_df]= df


### Ajout des tags et jointure avec revenus 

In [ ]:
dict_df_ref_unique_tag = {}
for nom_df,df in dict_df_ref_unique.items():
    nom_tag = nom_df.split("_")[2]
    df["adresse"] = df["adresse"].astype(str)
    df[f"adresse_has_{nom_tag}_init"] = False
    for i in range(len(df)):
        if nom_tag in df.at[i,"adresse"].split() :
            df.at[i,f"adresse_has_{nom_tag}_init"] = True 
    df = df.dropna(subset="CODE_IRIS")
    df_w_revenus = df.merge(df_revenus, how="left",left_on="CODE_IRIS",right_on="IRIS")
    dict_df_ref_unique_tag[nom_df] = df_w_revenus


### Ajout de la distance 

In [ ]:
list_id_ref_id = {}
dict_df_ref_unique_dist = {}
for nom_df,df in dict_df_ref_unique_tag.items():
    nom_tag = nom_df.split("_")[2]
    list_id_tag = []
    df_cleaned = df.copy()
    # PREP DATA 
    # df["DISP_MED20"] = df["DISP_MED20"].astype(float)
    # df_brute_rev["DISP_MED20"] = df_brute_rev["DISP_MED20"].astype(float)

    df_cleaned["y"] = df_cleaned["y"].astype(float)
    df_cleaned["x"] = df_cleaned["x"].astype(float)
    
    df_cleaned["geocodage_x"] = df_cleaned["geocodage_x_GPS"].astype(float)
    df_cleaned["geocodage_y"] = df_cleaned["geocodage_y_GPS"].astype(float)
    # df_brute_rev["x"] = df_brute_rev["x"].astype(float)
    # df_brute_rev["y"] = df_brute_rev["y"].astype(float)

    df_cleaned.dropna(subset=["x","y","geocodage_x_GPS","geocodage_y_GPS"], inplace=True)#,"DISP_MED20"])
    df_cleaned.reset_index(drop=True,inplace=True)
    #df_brute_rev = df_brute_rev.dropna(subset=["x","y"])#,"DISP_MED20"])

    # id_cleaned = df_cleaned.index 
    # id_cleaned_rev = df_brute_rev.index 
    # df_brute_rev_cleaned = df_brute_rev[(df_brute_rev.index.isin(id_cleaned)) & (df_brute_rev.index.isin(id_cleaned_rev))]
    # df_cleaned = df_cleaned[(df_cleaned.index.isin(id_cleaned)) & (df_cleaned.index.isin(id_cleaned_rev))]
    #df_cleaned[f"distance_{nom_tag}"] = 0.0 #df_cleaned[f"distance_{nom_tag}"].astype(float)
    for i in df_cleaned.index:
        if df_cleaned.at[i,f"adresse_has_{nom_tag}_init"] == True :
            list_id_tag.append(i)

        lon1 = pd.to_numeric(df_cleaned.at[i,"x"])
        lat1 = pd.to_numeric(df_cleaned.at[i,"y"])
        lon2 = pd.to_numeric(df_cleaned.at[i,"geocodage_x_GPS"])
        lat2 = pd.to_numeric(df_cleaned.at[i,"geocodage_y_GPS"])

        dist = calculer_distance_haversine(lat1, lon1, lat2, lon2)
        df_cleaned.loc[i,f"distance_{nom_tag}"] = dist
            
        #df_cleaned.loc[i,f"diff_revenu_{nom_tag}"] = pd.to_numeric(df_cleaned.at[i,"DISP_MED20"]) - pd.to_numeric(df_brute_rev_cleaned.at[i,"DISP_MED20"])
    dict_df_ref_unique_dist[nom_df]=df_cleaned
    list_id_ref_id[nom_tag] = list_id_tag 
    print(f"{nom_df} modifié")


## Biais joint

In [ ]:
files_ref_biais_join = [f for f in listdir("data_with_biais/ref_join_biais")]
dict_df_ref_join = {}

for file in files_ref_biais_join:
    nom_df = file[:-4]
    df = pd.read_csv(join("data_with_biais/ref_join_biais",file),sep=";",dtype=str)
    dict_df_ref_join[nom_df]= df

#### Ajout des tags

In [ ]:
dict_df_ref_join_tag = {}
for nom_df,df in dict_df_ref_join.items():
    nom_tag = nom_df.split("_")[2]+'_'+ nom_df.split("_")[3]
    tag_1 = nom_df.split("_")[2]
    tag_2 = nom_df.split("_")[3]
    df["adresse"] = df["adresse"].astype(str)
    df[f"adresse_has_{nom_tag}_init"] = False
    for i in range(len(df)):
        if tag_1 in df.at[i,"adresse"].split() or tag_2 in df.at[i,"adresse"].split():
            df.at[i,f"adresse_has_{nom_tag}_init"] = True 
    df = df.dropna(subset="CODE_IRIS")
    df_w_revenus = df.merge(df_revenus, how="left",left_on="CODE_IRIS",right_on="IRIS")
    dict_df_ref_join_tag[nom_df] = df_w_revenus


In [ ]:
list_id_ref_join = {}
dict_df_ref_join_dist = {}
for nom_df,df in dict_df_ref_join_tag.items():
    nom_tag = nom_df.split("_")[2]+'_'+ nom_df.split("_")[3]

    list_id_tag = []
    df_cleaned = df.copy()

    # PREP DATA 
    # df["DISP_MED20"] = df["DISP_MED20"].astype(float)
    # df_brute_rev["DISP_MED20"] = df_brute_rev["DISP_MED20"].astype(float)

    df_cleaned["y"] = df_cleaned["y"].astype(float)
    df_cleaned["x"] = df_cleaned["x"].astype(float)

    df_cleaned["geocodage_x"] = df_cleaned["geocodage_x"].astype(float)
    df_cleaned["geocodage_y"] = df_cleaned["geocodage_y"].astype(float)
    df_cleaned.dropna(subset=["x","y","geocodage_x_GPS","geocodage_y_GPS"], inplace=True)#,"DISP_MED20"])
    df_cleaned.reset_index(drop=True,inplace=True)
    # df_cleaned = df.dropna(subset=["x","y"])#,"DISP_MED20"])
    # df_brute_rev = df_brute_rev.dropna(subset=["x","y"])#,"DISP_MED20"])

    # id_cleaned = df_cleaned.index 
    # id_cleaned_rev = df_brute_rev.index 
    # df_brute_rev_cleaned = df_brute_rev[(df_brute_rev.index.isin(id_cleaned)) & (df_brute_rev.index.isin(id_cleaned_rev))]
    # df_cleaned = df_cleaned[(df_cleaned.index.isin(id_cleaned)) & (df_cleaned.index.isin(id_cleaned_rev))]


    for i in df_cleaned.index:
        if df_cleaned.at[i,f"adresse_has_{nom_tag}_init"] == True :
            list_id_tag.append(i)

        lon1 = pd.to_numeric(df_cleaned.at[i,"x"])
        lat1 = pd.to_numeric(df_cleaned.at[i,"y"])
        lon2 = pd.to_numeric(df_cleaned.at[i,"geocodage_x_GPS"])
        lat2 = pd.to_numeric(df_cleaned.at[i,"geocodage_y_GPS"])

        dist = calculer_distance_haversine(lat1, lon1, lat2, lon2)
        df_cleaned.loc[i,f"distance_{nom_tag}"] = dist
        # df_cleaned.loc[i,f"diff_revenu_{nom_tag}"] = pd.to_numeric(df_cleaned.at[i,"DISP_MED20"]) - pd.to_numeric(df_brute_rev_cleaned.at[i,"DISP_MED20"])

    dict_df_ref_join_dist[nom_df]=df_cleaned
    list_id_ref_join[nom_tag] = list_id_tag 
    print(f"{nom_df} modifié")


In [ ]:
dict_df_ref_join_dist["df_ref_APPT_BAT"]["distance_APPT_BAT"].isna().sum()

In [ ]:
for nom_df, df in dict_df_ref_join_dist.items():

    df.to_csv(f"data_with_biais/ref_join_biais/{nom_df}_dist.csv",sep=";")

# Analyse biais 

In [ ]:
dict_df_unique_dist["df_patients_APPT"].columns

In [ ]:
# percent_iris_shift_unique = {}
# resultats = []
# for nom_df, df in dict_df_unique_dist.items():
#     nom_tag = nom_df.split("_")[2]

#     df_iris_shift = df[df[f"diff_revenu_{nom_tag}"]!=0]

#     percent_iris_shift_unique[nom_tag] = len(df_iris_shift)/len(df) *100 

#     df["biais"] = nom_tag
#     df = df.rename(columns={
#         f"distance_{nom_tag}" : "distance"
#     })
#     resultats.append(df[["pseudo_provisoire","distance","biais"]])

# df_shift = pd.DataFrame.from_dict(percent_iris_shift_unique, orient="index")
# df_shift = df_shift.rename(columns={ 
#     df_shift.columns[0]: " % de changement d'iris "
# })

nom_tag = "APPT"
df = pd.read_csv("data_with_biais/patients_unique_biais/df_patients_APPT_dist.csv")
df_iris_shift = df[df[f"diff_revenu_{nom_tag}"]!=0]

test = len(df_iris_shift)/len(df) *100 

print(test)


In [ ]:
percent_iris_shift_join = {}

for nom_df, df in dict_df_join_dist.items():
    nom_tag = nom_df.split("_")[2] + "_" +nom_df.split("_")[3]

    df_iris_shift = df[df[f"diff_revenu_{nom_tag}"]!=0]

    percent_iris_shift_join[nom_tag] = len(df_iris_shift)/len(df) *100 

df_shift = pd.DataFrame.from_dict(percent_iris_shift_join, orient="index")
df_shift = df_shift.rename(columns={ 
    df_shift.columns[0]: " % de changement d'iris "
})
df_shift

In [ ]:
## Analyse dispersion biais : 

for nom_tag, id_biais in list_id_pat_unique.items():

    id_w_biais = list_id_pat_unique[nom_tag]
    nom_df ="df_patients_"+nom_tag

    df= dict_df_unique_dist[nom_df]

    df_biais = df[df.index.isin(id_w_biais)]
    df_brute_biais = df_brute[df_brute.index.isin(id_w_biais)]
    

In [ ]:
dist_biais = pd.concat(resultats, ignore_index=True)
dist_biais

In [ ]:
import seaborn as sns 

sns.set_style("whitegrid")


box_plot = sns.boxplot(x= "biais", y ="distance",data=dist_biais, showfliers = False)
box_plot.set(title="Distribution des distances à l'origine selon les biais pour nos patients (en km)")

ax = box_plot.axes
lines = ax.get_lines()
categories = ax.get_xticks()
cat = ax.get_xticklabels()
ax.set_xticklabels(cat,rotation=45)

box_plot.figure.tight_layout()

In [ ]:
percent_iris_shift_unique = {}
resultats_ref = []
for nom_df, df in dict_df_ref_unique_dist.items():
    nom_tag = nom_df.split("_")[2]

    #df_iris_shift = df[df[f"diff_revenu_{nom_tag}"]!=0]

    #percent_iris_shift_unique[nom_tag] = len(df_iris_shift)/len(df) *100 

    df["biais"] = nom_tag
    df = df.rename(columns={
        f"distance_{nom_tag}" : "distance"
    })
    resultats_ref.append(df[["identifiant_public_unite","distance","biais"]])

# df_shift = pd.DataFrame.from_dict(percent_iris_shift_unique, orient="index")
# df_shift = df_shift.rename(columns={ 
#     df_shift.columns[0]: " % de changement d'iris "
# })



In [ ]:
import seaborn as sns 
dist_ref = pd.concat(resultats_ref,ignore_index=True) 
sns.set_style("whitegrid")


box_plot = sns.boxplot(x= "biais", y ="distance",data=dist_ref, showfliers = False)
box_plot.set(title="Distribution des distances à l'origine selon les biais (en km) pour nos références")

ax = box_plot.axes
lines = ax.get_lines()
categories = ax.get_xticks()
cat = ax.get_xticklabels()
ax.set_xticklabels(cat,rotation=45)

box_plot.figure.tight_layout()

In [ ]:
import plotly.express as px
df = dict_df_ref_unique_dist["df_ref_APPT"]

df["score"] = df["score"].astype(float)
df["score"] = np.round(df["score"],4)

df_no_ref = df[(df["distance_APPT"]!=0)&(df["distance_APPT"]<100)]

fig = px.scatter(df_no_ref, x="score", y="distance_APPT")
fig.show()